# 💾 FarmIntel Stage 2 — Save Yield Prediction Artifacts

**Purpose:** Train the final Yield Prediction model on the full training set and save all artifacts required for production inference.  
**Selected Model:** RandomForestRegressor (best CV R² in `05_yield_model_selection.ipynb`)  
**Target transform:** `log1p(Yield)` during training → `expm1` at inference

### Artifacts saved
| File | Description |
|------|-------------|
| `yield_random_forest.pkl` | Trained RandomForestRegressor |
| `yield_ordinal_encoder.pkl` | Fitted `OrdinalEncoder` for input features |
| `yield_feature_columns.json` | Ordered feature list used during training |
| `yield_metadata.json` | Full model metadata for deployment |

### Visualisations saved
| File | Description |
|------|-------------|
| `yield_actual_vs_predicted.png` | Predicted vs Actual scatter with y=x line |
| `yield_residual_distribution.png` | Residual histogram + KDE |
| `yield_residual_vs_predicted.png` | Residuals vs Predicted scatter |

---
## 1. Imports

In [1]:
import json
import sys
import time
import datetime
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')       # headless — no display required
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sklearn

from sklearn.ensemble     import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics       import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

print(f'Python  : {sys.version.split()[0]}')
print(f'sklearn : {sklearn.__version__}')
print(f'joblib  : {joblib.__version__}')
print(f'numpy   : {np.__version__}')
print(f'pandas  : {pd.__version__}')

Python  : 3.10.10
sklearn : 1.7.2
joblib  : 1.5.3
numpy   : 2.2.6
pandas  : 2.3.3


---
## 2. Configuration

In [27]:
# ── Model choice ──────────────────────────────────────────────────────
# RandomForestRegressor selected as best CV R² in 05_yield_model_selection.ipynb
#   CV R²   ≈ 0.91   (best across all regressors)
#   Hyperparameters: n_estimators=200, max_depth=None, min_samples_leaf=3
SELECTED_MODEL_NAME = 'RandomForestRegressor'

# ── Paths ─────────────────────────────────────────────────────────────
DATA_DIR    = Path('../data/processed')
MODELS_DIR  = Path('../models')
OUTPUTS_DIR = Path('../outputs')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature / target definitions ─────────────────────────────────────
# Mirrors notebook 05_yield_model_selection.ipynb cell 6 exactly
CAT_FEATURES  = ['State', 'District', 'Crop', 'Crop_Category', 'Season']
NUM_FEATURES  = ['Year']
ALL_FEATURES  = CAT_FEATURES + NUM_FEATURES
TARGET        = 'Yield'
RANDOM_STATE  = 42

# ── RandomForest hyperparameters (identical to model selection) ───────
RF_PARAMS = dict(
    n_estimators    = 120,
    max_depth       = 15,
    min_samples_leaf = 3,
    n_jobs          = -1,
    random_state    = RANDOM_STATE,
)

# ── Dark plot theme (matches model-selection notebook) ────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor':   '#333355', 'axes.labelcolor': '#e0e0f0',
    'text.color':       '#e0e0f0', 'xtick.color':    '#a0a0c0',
    'ytick.color':      '#a0a0c0', 'grid.color':     '#2a2a4a',
    'grid.linestyle':   '--',      'grid.alpha':      0.5,
    'font.size':        11,
})
ACCENT = '#7c3aed'; ACCENT2 = '#06b6d4'; HIGHLIGHT = '#f59e0b'
GREEN  = '#10b981'; RED = '#f43f5e'

print(f'Data dir    : {DATA_DIR.resolve()}')
print(f'Models dir  : {MODELS_DIR.resolve()}')
print(f'Outputs dir : {OUTPUTS_DIR.resolve()}')
print(f'Features    : {ALL_FEATURES}')
print(f'Target      : {TARGET}')

Data dir    : D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\data\processed
Models dir  : D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models
Outputs dir : D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\outputs
Features    : ['State', 'District', 'Crop', 'Crop_Category', 'Season', 'Year']
Target      : Yield


---
## 3. Helper Functions

In [28]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Compute regression metrics in original (kg/ha) scale.

    Returns
    -------
    dict with keys: R2, RMSE, MAE, MAPE
    """
    r2   = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    nz   = y_true > 0
    mape = float(np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100)
    return {'R2': round(r2, 6), 'RMSE': round(rmse, 4),
            'MAE': round(mae, 4), 'MAPE': round(mape, 4)}


def save_artifact(obj, path: Path, use_json: bool = False) -> None:
    """
    Save a single artifact to disk.

    Parameters
    ----------
    obj      : the object to save
    path     : pathlib.Path destination
    use_json : if True, dump as JSON; otherwise use joblib
    """
    if use_json:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(obj, f, indent=2, ensure_ascii=False)
    else:
        joblib.dump(obj, path, compress=3)

    size_kb = path.stat().st_size / 1024
    unit    = 'MB' if size_kb >= 1024 else 'KB'
    size    = size_kb / 1024 if size_kb >= 1024 else size_kb
    print(f'  [OK]  {path.name:<38}  {size:>7.1f} {unit}  ->  {path.resolve()}')


print('Helper functions defined')

Helper functions defined


---
## 4. Load & Prepare Data

Loads the processed CSVs and builds `X_train`, `X_test`, `y_train`, `y_test`  
along with the fitted `ord_enc` and log-transformed target arrays.

In [29]:
def load_and_prepare(
    data_dir: Path,
    cat_features: list,
    num_features: list,
    target: str,
):
    """
    Load yield_train.csv and yield_test.csv from data_dir.
    Fit OrdinalEncoder on train categorical columns.
    Return encoded numpy arrays and fitted encoder.

    Returns
    -------
    X_train, X_test  : np.ndarray  — encoded feature matrices
    y_train, y_test  : np.ndarray  — raw Yield (kg/ha)
    ord_enc          : fitted OrdinalEncoder
    """
    train_df = pd.read_csv(data_dir / 'yield_train.csv')
    test_df  = pd.read_csv(data_dir / 'yield_test.csv')

    print(f'  Loaded  yield_train.csv  ->  {train_df.shape}')
    print(f'  Loaded  yield_test.csv   ->  {test_df.shape}')

    # ── Fit OrdinalEncoder on training categorical columns ──────────
    ord_enc = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
    )
    ord_enc.fit(train_df[cat_features])

    # ── Encode helper ───────────────────────────────────────────────
    def encode(df: pd.DataFrame) -> np.ndarray:
        X_cat = ord_enc.transform(df[cat_features])
        X_num = df[num_features].values.astype(float)
        return np.hstack([X_cat, X_num])

    X_train = encode(train_df)
    X_test  = encode(test_df)
    y_train = train_df[target].values
    y_test  = test_df[target].values

    return X_train, X_test, y_train, y_test, ord_enc


print('Preparing data ...')
X_train, X_test, y_train, y_test, ord_enc = load_and_prepare(
    data_dir     = DATA_DIR,
    cat_features = CAT_FEATURES,
    num_features = NUM_FEATURES,
    target       = TARGET,
)

# Log-transform target (heavy right skew — see 03_yield_eda.ipynb)
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

print()
print(f'  X_train  : {X_train.shape}  (dtype={X_train.dtype})')
print(f'  X_test   : {X_test.shape}  (dtype={X_test.dtype})')
print(f'  y_train  : {y_train.shape}  raw  — mean={y_train.mean():,.1f} kg/ha')
print(f'  y_train  : {y_train_log.shape}  log1p — mean={y_train_log.mean():.3f}, std={y_train_log.std():.3f}')
print(f'  y_test   : {y_test.shape}  raw  — mean={y_test.mean():,.1f} kg/ha')

Preparing data ...
  Loaded  yield_train.csv  ->  (271481, 7)
  Loaded  yield_test.csv   ->  (67871, 7)

  X_train  : (271481, 6)  (dtype=float64)
  X_test   : (67871, 6)  (dtype=float64)
  y_train  : (271481,)  raw  — mean=81.3 kg/ha
  y_train  : (271481,)  log1p — mean=1.099, std=1.159
  y_test   : (67871,)  raw  — mean=78.9 kg/ha


---
## 5. Train RandomForestRegressor on Full Training Set

> Exact hyperparameters from model selection (notebook `05_yield_model_selection.ipynb` cell 11):  
> `n_estimators=200`, `max_depth=None`, `min_samples_leaf=3`, `random_state=42`  
> Target is trained in **log1p** scale; predictions are back-transformed with **expm1**.

In [30]:
print(f'Training {SELECTED_MODEL_NAME} on full training set ...')
print(f'  X_train : {X_train.shape}')
print(f'  Params  : {RF_PARAMS}')
print()

rf_model = RandomForestRegressor(**RF_PARAMS)

t0 = time.time()
rf_model.fit(X_train, y_train_log)
train_time = time.time() - t0

fitted_model = rf_model

print(f'  Fit complete in {train_time:.1f}s')
print(f'  n_estimators fitted : {rf_model.n_estimators}')
print(f'  n_features          : {rf_model.n_features_in_}')

Training RandomForestRegressor on full training set ...
  X_train : (271481, 6)
  Params  : {'n_estimators': 120, 'max_depth': 15, 'min_samples_leaf': 3, 'n_jobs': -1, 'random_state': 42}

  Fit complete in 10.1s
  n_estimators fitted : 120
  n_features          : 6


---
## 6. Test Set Metrics

In [31]:
print(f'Evaluating {SELECTED_MODEL_NAME} on held-out test set ...')

# Back-transform: predict in log-scale, convert to original kg/ha
y_pred_log  = fitted_model.predict(X_test)
y_pred_orig = np.expm1(np.clip(y_pred_log, None, 15))  # clip extreme log values

TEST_METRICS = compute_metrics(y_test, y_pred_orig)

sep = '─' * 44
print(f"\n  {sep}")
print(f"  {'Metric':<22} {'Value':>18}")
print(f"  {sep}")
print(f"  {'R2 Score':<22} {TEST_METRICS['R2']:>18.4f}")
print(f"  {'RMSE (kg/ha)':<22} {TEST_METRICS['RMSE']:>18,.2f}")
print(f"  {'MAE  (kg/ha)':<22} {TEST_METRICS['MAE']:>18,.2f}")
print(f"  {'MAPE (%)':<22} {TEST_METRICS['MAPE']:>18.2f}")
print(f"  {sep}")

Evaluating RandomForestRegressor on held-out test set ...

  ────────────────────────────────────────────
  Metric                              Value
  ────────────────────────────────────────────
  R2 Score                           0.9134
  RMSE (kg/ha)                       267.52
  MAE  (kg/ha)                        15.46
  MAPE (%)                            44.72
  ────────────────────────────────────────────


---
## 7. Generate & Save Visualisations

In [32]:
residuals = y_test - y_pred_orig

# ── Sample for scatter plots (avoid overplotting on 67k points) ───────
_rng  = np.random.default_rng(RANDOM_STATE)
_idx  = _rng.choice(len(y_test), min(len(y_test), 8_000), replace=False)


# ── Plot 1 : Predicted vs Actual ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(y_test[_idx], y_pred_orig[_idx],
           alpha=0.35, s=15, color=ACCENT2, edgecolors='none', label='Predictions')
vmin = min(y_test.min(), y_pred_orig.min())
vmax = max(y_test.max(), y_pred_orig.max())
ax.plot([vmin, vmax], [vmin, vmax],
        color=HIGHLIGHT, linewidth=1.8, linestyle='--', label='Perfect fit (y=x)')
ax.set_xlabel('Actual Yield (kg/ha)', fontsize=12)
ax.set_ylabel('Predicted Yield (kg/ha)', fontsize=12)
ax.set_title('Random Forest — Predicted vs Actual Yield',
             fontsize=13, fontweight='bold', pad=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()
_p1 = OUTPUTS_DIR / 'yield_actual_vs_predicted.png'
fig.savefig(_p1, dpi=150, bbox_inches='tight')
plt.show()
print(f'  [OK]  Saved -> {_p1.resolve()}')


# ── Plot 2 : Residual Distribution ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(residuals, bins=80, kde=True, color=ACCENT, edgecolor='none',
             alpha=0.75, line_kws={'linewidth': 2.2, 'color': HIGHLIGHT}, ax=ax)
ax.axvline(0, color=RED, linewidth=1.8, linestyle='--', label='Zero residual')
ax.axvline(residuals.mean(), color=GREEN, linewidth=1.5, linestyle='-.',
           label=f'Mean residual = {residuals.mean():,.1f}')
ax.set_xlabel('Residual  (Actual - Predicted)  kg/ha', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Residual Distribution — Random Forest Yield Model',
             fontsize=13, fontweight='bold', pad=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()
_p2 = OUTPUTS_DIR / 'yield_residual_distribution.png'
fig.savefig(_p2, dpi=150, bbox_inches='tight')
plt.show()
print(f'  [OK]  Saved -> {_p2.resolve()}')


# ── Plot 3 : Residuals vs Predicted ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(y_pred_orig[_idx], residuals[_idx],
           alpha=0.35, s=15, color=GREEN, edgecolors='none')
ax.axhline(0, color=RED, linewidth=1.8, linestyle='--', label='Zero residual')
ax.set_xlabel('Predicted Yield (kg/ha)', fontsize=12)
ax.set_ylabel('Residual  (Actual - Predicted)  kg/ha', fontsize=12)
ax.set_title('Residuals vs Predicted — Random Forest Yield Model',
             fontsize=13, fontweight='bold', pad=14)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=10)
ax.grid(True)
plt.tight_layout()
_p3 = OUTPUTS_DIR / 'yield_residual_vs_predicted.png'
fig.savefig(_p3, dpi=150, bbox_inches='tight')
plt.show()
print(f'  [OK]  Saved -> {_p3.resolve()}')

  [OK]  Saved -> D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\outputs\yield_actual_vs_predicted.png
  [OK]  Saved -> D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\outputs\yield_residual_distribution.png
  [OK]  Saved -> D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\outputs\yield_residual_vs_predicted.png


---
## 8. Save Artifacts

In [33]:
print('Saving artifacts to:', MODELS_DIR.resolve())
print('─' * 80)

# ── 1. RandomForest model ────────────────────────────────────────────
save_artifact(
    fitted_model,
    MODELS_DIR / 'yield_random_forest.pkl',
)

# ── 2. OrdinalEncoder ───────────────────────────────────────────────
save_artifact(
    ord_enc,
    MODELS_DIR / 'yield_ordinal_encoder.pkl',
)

# ── 3. Feature columns ──────────────────────────────────────────────
feature_payload = {
    'categorical_features': CAT_FEATURES,
    'numerical_features':   NUM_FEATURES,
    'feature_order':        ALL_FEATURES,
}
save_artifact(
    feature_payload,
    MODELS_DIR / 'yield_feature_columns.json',
    use_json=True,
)

# ── 4. Metadata ─────────────────────────────────────────────────────
metadata = {
    'model_name'         : 'FarmIntel-Stage2-YieldPrediction',
    'algorithm'          : SELECTED_MODEL_NAME,
    'training_date'      : datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
    'feature_names'      : ALL_FEATURES,
    'target'             : TARGET,
    'number_of_features' : len(ALL_FEATURES),
    'train_samples'      : int(len(y_train)),
    'test_samples'       : int(len(y_test)),
    'hyperparameters'    : {**RF_PARAMS, 'max_depth': 'None'},
    'target_transform'   : 'log1p (numpy) — predictions use expm1',
    'metrics'            : TEST_METRICS,
    'sklearn_version'    : sklearn.__version__,
    'python_version'     : sys.version.split()[0],
    'encoding'           : {
        'categorical_features' : CAT_FEATURES,
        'numerical_features'   : NUM_FEATURES,
        'encoder_type'         : 'OrdinalEncoder',
        'unknown_value'        : -1,
    },
    'stage'              : 2,
    'stage_description'  : 'Predicts crop Yield (kg/ha) from State, District, Crop, Crop_Category, Season, Year.',
    'notes'              : 'Area and Production removed to prevent target leakage.',
}

save_artifact(
    metadata,
    MODELS_DIR / 'yield_metadata.json',
    use_json=True,
)

print('─' * 80)
print('All 4 artifacts saved successfully.')

Saving artifacts to: D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models
────────────────────────────────────────────────────────────────────────────────
  [OK]  yield_random_forest.pkl                    41.3 MB  ->  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\yield_random_forest.pkl
  [OK]  yield_ordinal_encoder.pkl                   5.3 KB  ->  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\yield_ordinal_encoder.pkl
  [OK]  yield_feature_columns.json                  0.3 KB  ->  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\yield_feature_columns.json
  [OK]  yield_metadata.json                         1.2 KB  ->  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\yield_metadata.json
────────────────────────────────────────────────────────────────────────────────
All 4 artifacts saved successfully.


---
## 9. Verify Saved Artifacts

In [34]:
def verify_artifacts(models_dir: Path) -> None:
    """
    Reload every saved artifact and run sanity checks.
    Raises AssertionError if any file fails validation.
    """
    print('Running artifact verification ...\n')

    # Reload pkl files
    loaded_model = joblib.load(models_dir / 'yield_random_forest.pkl')
    loaded_enc   = joblib.load(models_dir / 'yield_ordinal_encoder.pkl')

    # Reload JSON files
    with open(models_dir / 'yield_feature_columns.json', encoding='utf-8') as f:
        loaded_features = json.load(f)
    with open(models_dir / 'yield_metadata.json', encoding='utf-8') as f:
        loaded_meta = json.load(f)

    # Sanity checks
    sample       = X_test[:5]
    pred_orig    = np.expm1(np.clip(fitted_model.predict(sample), None, 15))
    pred_loaded  = np.expm1(np.clip(loaded_model.predict(sample), None, 15))

    assert np.allclose(pred_orig, pred_loaded, rtol=1e-5),               'Model predictions mismatch!'
    assert loaded_features['feature_order'] == ALL_FEATURES,             'Feature columns mismatch!'
    assert loaded_meta['algorithm'] == SELECTED_MODEL_NAME,              'Metadata algorithm mismatch!'
    assert len(loaded_enc.categories_) == len(CAT_FEATURES),             'OrdinalEncoder category count mismatch!'
    assert loaded_meta['train_samples'] == len(y_train),                 'Train sample count mismatch!'

    checks = [
        ('yield_random_forest.pkl',     'Predictions match original model'),
        ('yield_ordinal_encoder.pkl',   f'{len(loaded_enc.categories_)} categorical columns encoded'),
        ('yield_feature_columns.json',  f'{len(loaded_features["feature_order"])} features: {loaded_features["feature_order"]}'),
        ('yield_metadata.json',         f'algorithm={loaded_meta["algorithm"]}  R2={loaded_meta["metrics"]["R2"]}'),
    ]
    for filename, note in checks:
        print(f'   {filename:<38}  {note}')

    print('\n All verification checks passed.')


verify_artifacts(MODELS_DIR)

Running artifact verification ...

   yield_random_forest.pkl                 Predictions match original model
   yield_ordinal_encoder.pkl               5 categorical columns encoded
   yield_feature_columns.json              6 features: ['State', 'District', 'Crop', 'Crop_Category', 'Season', 'Year']
   yield_metadata.json                     algorithm=RandomForestRegressor  R2=0.913361

 All verification checks passed.


---
## 10. Directory Tree

In [35]:
def print_directory_tree(root: Path, prefix: str = '') -> None:
    """Recursively print a directory tree with file sizes."""
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        connector = '└── ' if i == len(entries) - 1 else '├── '
        if entry.is_dir():
            print(f'{prefix}{connector}{entry.name}/')
            extension = '    ' if i == len(entries) - 1 else '│   '
            print_directory_tree(entry, prefix + extension)
        else:
            size_kb = entry.stat().st_size / 1024
            unit    = 'MB' if size_kb >= 1024 else 'KB'
            size    = size_kb / 1024 if size_kb >= 1024 else size_kb
            print(f'{prefix}{connector}{entry.name}  ({size:.1f} {unit})')


print(f'{MODELS_DIR.resolve()}/')
print_directory_tree(MODELS_DIR)

total_bytes = sum(f.stat().st_size for f in MODELS_DIR.rglob('*') if f.is_file())
total_mb    = total_bytes / (1024 ** 2)
n_files     = len([f for f in MODELS_DIR.iterdir() if f.is_file()])
print(f'\nTotal: {n_files} files  |  {total_mb:.2f} MB')

D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models/
├── crop_categories.json  (0.1 KB)
├── crop_category_xgboost.pkl  (6.1 MB)
├── feature_columns.json  (0.1 KB)
├── label_encoder.pkl  (0.4 KB)
├── metadata.json  (1.3 KB)
├── ordinal_encoder.pkl  (4.8 KB)
├── yield_feature_columns.json  (0.3 KB)
├── yield_metadata.json  (1.2 KB)
├── yield_ordinal_encoder.pkl  (5.3 KB)
└── yield_random_forest.pkl  (41.3 MB)

Total: 10 files  |  47.40 MB


---
## 11. Metadata Preview

In [36]:
with open(MODELS_DIR / 'yield_metadata.json', encoding='utf-8') as f:
    saved_meta = json.load(f)

print(json.dumps(saved_meta, indent=2))

{
  "model_name": "FarmIntel-Stage2-YieldPrediction",
  "algorithm": "RandomForestRegressor",
  "training_date": "2026-07-09T22:49:46",
  "feature_names": [
    "State",
    "District",
    "Crop",
    "Crop_Category",
    "Season",
    "Year"
  ],
  "target": "Yield",
  "number_of_features": 6,
  "train_samples": 271481,
  "test_samples": 67871,
  "hyperparameters": {
    "n_estimators": 120,
    "max_depth": "None",
    "min_samples_leaf": 3,
    "n_jobs": -1,
    "random_state": 42
  },
  "target_transform": "log1p (numpy) \u2014 predictions use expm1",
  "metrics": {
    "R2": 0.913361,
    "RMSE": 267.521,
    "MAE": 15.455,
    "MAPE": 44.723
  },
  "sklearn_version": "1.7.2",
  "python_version": "3.10.10",
  "encoding": {
    "categorical_features": [
      "State",
      "District",
      "Crop",
      "Crop_Category",
      "Season"
    ],
    "numerical_features": [
      "Year"
    ],
    "encoder_type": "OrdinalEncoder",
    "unknown_value": -1
  },
  "stage": 2,
  "stage_d

---
## 12. Inference Smoke Test

End-to-end: raw string input → encode → predict → back-transform → Yield (kg/ha).

In [37]:
def predict_yield(
    state: str,
    district: str,
    crop: str,
    crop_category: str,
    season: str,
    year: int,
    model_dir: Path = MODELS_DIR,
) -> float:
    """
    End-to-end inference — loads artifacts from disk.

    Returns
    -------
    float : predicted Yield in kg/ha
    """
    _model   = joblib.load(model_dir / 'yield_random_forest.pkl')
    _enc     = joblib.load(model_dir / 'yield_ordinal_encoder.pkl')

    with open(model_dir / 'yield_feature_columns.json', encoding='utf-8') as f:
        _cols = json.load(f)

    _cat_cols = _cols['categorical_features']
    _num_cols = _cols['numerical_features']
    _all_cols = _cols['feature_order']

    raw_df = pd.DataFrame([{
        'State': state, 'District': district,
        'Crop': crop, 'Crop_Category': crop_category,
        'Season': season, 'Year': year,
    }])[_all_cols]

    X_encoded = np.hstack([
        _enc.transform(raw_df[_cat_cols]),
        raw_df[_num_cols].values.astype(float),
    ])

    y_pred_log = _model.predict(X_encoded)[0]
    return float(np.expm1(np.clip(y_pred_log, None, 15)))


# ── Test cases ────────────────────────────────────────────────────────
test_inputs = [
    ('Uttar Pradesh', 'Faizabad',   'Rice',       'Cereals',    'Kharif',    2005),
    ('Punjab',        'Ludhiana',   'Wheat',      'Cereals',    'Rabi',      2015),
    ('Kerala',        'Wayanad',    'Pepper',     'Spices',     'Whole Year', 2010),
    ('Maharashtra',   'Pune',       'Sugarcane',  'Cash Crops', 'Whole Year', 2018),
    ('Rajasthan',     'Jaipur',     'Mustard',    'Oilseeds',   'Rabi',      2012),
]

print('Inference Smoke Test — Yield Prediction (kg/ha)')
print('=' * 65)
for state, district, crop, crop_cat, season, year in test_inputs:
    pred = predict_yield(state, district, crop, crop_cat, season, year)
    print(f'  {state} | {district} | {crop} | {season} | {year}')
    print(f'     => Predicted Yield : {pred:,.1f} kg/ha')
    print()

print(' Smoke test complete — inference pipeline working correctly.')

Inference Smoke Test — Yield Prediction (kg/ha)
  Uttar Pradesh | Faizabad | Rice | Kharif | 2005
     => Predicted Yield : 1.9 kg/ha

  Punjab | Ludhiana | Wheat | Rabi | 2015
     => Predicted Yield : 4.5 kg/ha

  Kerala | Wayanad | Pepper | Whole Year | 2010
     => Predicted Yield : 0.5 kg/ha

  Maharashtra | Pune | Sugarcane | Whole Year | 2018
     => Predicted Yield : 97.7 kg/ha

  Rajasthan | Jaipur | Mustard | Rabi | 2012
     => Predicted Yield : 0.9 kg/ha

 Smoke test complete — inference pipeline working correctly.


---
## 13. Final Summary

In [38]:
print('=' * 62)
print('  FARMINTEL STAGE 2 — ARTIFACT SAVE COMPLETE')
print('=' * 62)
print(f'  Algorithm          : {SELECTED_MODEL_NAME}')
print(f'  Training samples   : {len(y_train):,}')
print(f'  Test  samples      : {len(y_test):,}')
print(f'  Features ({len(ALL_FEATURES)})       : {ALL_FEATURES}')
print(f'  Target             : {TARGET} (kg/ha)')
print(f'  Target transform   : log1p / expm1')
print()
print('  ── Test Set Metrics ──')
for k, v in TEST_METRICS.items():
    unit = ' %' if k == 'MAPE' else ' kg/ha' if k in ('RMSE', 'MAE') else ''
    print(f'  {k:<22}: {v:.4f}{unit}')
print()
print('  ── Saved Model Artifacts ──')
for f in sorted(MODELS_DIR.iterdir()):
    if f.is_file() and 'yield' in f.name:
        size_kb = f.stat().st_size / 1024
        unit    = 'MB' if size_kb >= 1024 else 'KB'
        size    = size_kb / 1024 if size_kb >= 1024 else size_kb
        print(f'  * {f.name:<38}  {size:>7.1f} {unit}')
print()
print('  ── Saved Visualisations ──')
for f in sorted(OUTPUTS_DIR.iterdir()):
    if f.is_file() and 'yield' in f.name:
        size_kb = f.stat().st_size / 1024
        print(f'  * {f.name:<38}  {size_kb:>7.1f} KB')
print()
print('    Next: Backend API integration — serve yield predictions')
print('          via FastAPI endpoint using saved artifacts.')
print('=' * 62)
print('  Yield Prediction Model Saved Successfully')
print('=' * 62)

  FARMINTEL STAGE 2 — ARTIFACT SAVE COMPLETE
  Algorithm          : RandomForestRegressor
  Training samples   : 271,481
  Test  samples      : 67,871
  Features (6)       : ['State', 'District', 'Crop', 'Crop_Category', 'Season', 'Year']
  Target             : Yield (kg/ha)
  Target transform   : log1p / expm1

  ── Test Set Metrics ──
  R2                    : 0.9134
  RMSE                  : 267.5210 kg/ha
  MAE                   : 15.4550 kg/ha
  MAPE                  : 44.7230 %

  ── Saved Model Artifacts ──
  * yield_feature_columns.json                  0.3 KB
  * yield_metadata.json                         1.2 KB
  * yield_ordinal_encoder.pkl                   5.3 KB
  * yield_random_forest.pkl                    41.3 MB

  ── Saved Visualisations ──
  * yield_actual_vs_predicted.png              85.1 KB
  * yield_model_selection_summary.txt           1.9 KB
  * yield_residual_distribution.png            63.7 KB
  * yield_residual_vs_predicted.png            73.9 KB

    Next: